# Introduction

In this notebook, we want to understand how to evaluate text representations through text classification. Basically, we compare different embedding models by using the same modeling approach and parameters to determine which embeddings perform better.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Dataset

We use a sentiment review dataset and filter it to include only extreme sentiment categories:

- Very Bad Reviews (Rating: 1) - Negative class
- Very Good Reviews (Rating: 5) - Positive class

This creates a binary classification problem with clear sentiment distinction, allowing us to better evaluate the effectiveness of different text representation methods.

In [4]:
df = pd.read_csv("/content/drive/MyDrive/Purwadhika/Text Representation/data/gojek_review.csv")
df = df[df.rate.isin([1, 5])] # Only classify very bad and very good rating
df.head()

,review,rate
0,Sangat kecewa. Kecewa sekali. Udh top up. Mau ...,1
1,Ga niat ngasih promo sialan temen udh pake ref...,1
2,Kalau sistemnya rata begini apa bedanya yg raj...,1
3,"Ongkosnya da mahal, minimal 16rb..... Sekarang...",1
4,Tolol anjing..!!!! Aplikasi yang katanga karya...,1


# Dataset Splitting

The filtered dataset is split into training and testing sets to ensure proper model evaluation:

- Training set: Used to train the classification models
- Test set: Used to evaluate model performance and compare different embedding methods

In [5]:
X = df.review
y = df.rate

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((763,), (191,), (763,), (191,))

In [6]:
from sklearn.ensemble import RandomForestClassifier

# Evaluation Approach

All methods use the same classification model and parameters to ensure fair comparison. Performance metrics will help determine which text representation method works best for sentiment classification on this dataset.



# Method 1: Bag of Words

Basic text representation that converts text into numerical vectors based on word frequency:

- Creates a vocabulary of unique words from the training corpus
- Represents each document as a vector where each dimension corresponds to a word's frequency
- Simple but effective baseline method that ignores word order

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

In [8]:
vectorizer = CountVectorizer()
X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [9]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.8167539267015707



# Method 2: Bag of Words + NGRAM + Preprocessing

Enhanced version of BoW that captures some contextual information:

- Includes unigrams (single words) and bigrams/trigrams (word sequences)
- Helps capture phrase-level meanings and local word dependencies
- Provides richer text representation compared to basic BoW

In [10]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
import nltk
nltk.download('stopwords')
sw_indo = stopwords.words("indonesian") + list(punctuation)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words=sw_indo)
X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


In [12]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.8010471204188482



# Method 3: TFIDF

Statistical measure that reflects word importance in documents:

- Weighs terms based on their frequency in a document and rarity across the corpus
- Reduces the impact of common words while emphasizing distinctive terms
- Often performs better than raw frequency counts for classification tasks

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
vectorizer = TfidfVectorizer()
X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [15]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.8324607329842932



# Method 4: Word2Vec

Dense vector representation that captures semantic relationships:

- Pre-trained or custom-trained word embeddings that represent words in continuous vector space
- Captures semantic similarity between words based on contextual usage
- Document representation typically created by averaging word vectors

In [22]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 21.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.0
    Uninstalling scipy-1.16.0:
      Successfully uninstalled scipy-1.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatibl

In [18]:
from gensim.models import Word2Vec
import numpy as np
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [20]:
sentences = [word_tokenize(rev.lower()) for rev in X_train]
model = Word2Vec(sentences=sentences, vector_size=128, window=5, min_count=3, workers=4, epochs=1000)
w2v = model.wv

In [21]:
vecs = []
for sentence in X_train:
    word_vector = np.zeros(128)
    for word in sentence:
        if word not in sw_indo and word in w2v.index_to_key:
            word_vector += w2v[word]
    vecs.append(word_vector)
X_train_vector = np.array(vecs)

In [22]:
vecs = []
for sentence in X_test:
    word_vector = np.zeros(128)
    for word in sentence:
        if word not in sw_indo and word in w2v.index_to_key:
            word_vector += w2v[word]
    vecs.append(word_vector)
X_test_vector = np.array(vecs)

In [23]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.7539267015706806



# Method 5: FastText

Advanced word embedding method that handles out-of-vocabulary words:

- Extension of Word2Vec that considers subword information (character n-grams)
- Better handling of rare words, misspellings, and morphological variations
- Particularly effective for languages with rich morphology

In [24]:
from gensim.models import FastText

In [32]:
sentences = [word_tokenize(rev.lower()) for rev in X_train]
model = FastText(sentences=sentences, vector_size=128, window=5, min_count=3, workers=4, epochs=1000)
w2v = model.wv

In [33]:
vecs = []
for sentence in X_train:
    word_vector = np.zeros(128)
    for word in sentence:
        if word not in sw_indo and word in w2v.index_to_key:
            word_vector += w2v[word]
    vecs.append(word_vector)
X_train_vector = np.array(vecs)

In [34]:
vecs = []
for sentence in X_test:
    word_vector = np.zeros(128)
    for word in sentence:
        if word not in sw_indo and word in w2v.index_to_key:
            word_vector += w2v[word]
    vecs.append(word_vector)
X_test_vector = np.array(vecs)

In [28]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.7382198952879581



# Method 6: GloVe (Global Vectors for Word Representation)

Advanced word embedding method that combines global matrix factorization with local context window methods:

- Leverages global word co-occurrence statistics from the entire corpus rather than just local context windows
- Trains on aggregated global word-word co-occurrence matrix to capture semantic relationships
- Bridges the gap between count-based methods (like LSA) and predictive methods (like Word2Vec)

In [44]:
import gensim.downloader as api

# Download pre-trained GloVe vectors through Gensim
glove_vectors = api.load("glove-wiki-gigaword-300")

[==================================================] 100.0% 376.1/376.1MB downloaded


In [47]:
# Convert training sentences to vectors
sentences_train = [word_tokenize(rev.lower()) for rev in X_train]
vecs_train = []
for sentence in sentences_train:
    word_vector = np.zeros(300)  # GloVe vectors are 300-dimensional
    word_count = 0
    for word in sentence:
        if word not in sw_indo and word in glove_vectors:
            word_vector += glove_vectors[word]
            word_count += 1
    if word_count > 0:
        word_vector = word_vector / word_count  # Average instead of sum
    vecs_train.append(word_vector)
X_train_vector = np.array(vecs_train)

In [48]:
# Convert test sentences to vectors
vecs_test = []
for sentence in X_test:
    word_vector = np.zeros(300)  # GloVe vectors are 300-dimensional
    word_count = 0
    for word in sentence:
        if word not in sw_indo and word in glove_vectors:
            word_vector += glove_vectors[word]
            word_count += 1
    if word_count > 0:
        word_vector = word_vector / word_count  # Average instead of sum
    vecs_test.append(word_vector)
X_test_vector = np.array(vecs_test)

In [49]:
# Train classifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 0.9986893840104849
Test: 0.7486910994764397



# Conclusion

Based on the test set, the model with the best performance on this dataset is TF-IDF. There are several factors that might explain this result:

1. The prediction algorithm (Random Forest) that is responsible for model prediction may work particularly well with TF-IDF features
2. The amount of training data - usually Word2Vec, Gensim, and FastText perform better when there is much more training data available
3. The use case of sentiment analysis - basically we just need to capture important keywords rather than sentence context, because usually we can predict sentiment from only a few key words